In [7]:
import sys, random, math
from collections import Counter
import numpy as np
from tensorflow.keras.datasets import imdb

# Load IMDb dataset with top 10000 words
(X_train, X_test), (y_train, y_test) = imdb.load_data(num_words=10000)

# Vocabulary is range(1000)
vocab = list(range(1000))

# Filter tokens to only include words within vocab (index < 1000)
tokens = []
for review in X_train:
    filtered = [w for w in review if w < 1000]
    if len(filtered) > 1:
        tokens.append(filtered)

print(tokens[0:3])

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=0)

[[1, 14, 22, 16, 43, 530, 973, 65, 458, 66, 4, 173, 36, 256, 5, 25, 100, 43, 838, 112, 50, 670, 2, 9, 35, 480, 284, 5, 150, 4, 172, 112, 167, 2, 336, 385, 39, 4, 172, 17, 546, 38, 13, 447, 4, 192, 50, 16, 6, 147, 19, 14, 22, 4, 469, 4, 22, 71, 87, 12, 16, 43, 530, 38, 76, 15, 13, 4, 22, 17, 515, 17, 12, 16, 626, 18, 2, 5, 62, 386, 12, 8, 316, 8, 106, 5, 4, 16, 480, 66, 33, 4, 130, 12, 16, 38, 619, 5, 25, 124, 51, 36, 135, 48, 25, 33, 6, 22, 12, 215, 28, 77, 52, 5, 14, 407, 16, 82, 2, 8, 4, 107, 117, 15, 256, 4, 2, 7, 5, 723, 36, 71, 43, 530, 476, 26, 400, 317, 46, 7, 4, 2, 13, 104, 88, 4, 381, 15, 297, 98, 32, 56, 26, 141, 6, 194, 18, 4, 226, 22, 21, 134, 476, 26, 480, 5, 144, 30, 18, 51, 36, 28, 224, 92, 25, 104, 4, 226, 65, 16, 38, 88, 12, 16, 283, 5, 16, 113, 103, 32, 15, 16, 19, 178, 32], [1, 194, 194, 78, 228, 5, 6, 134, 26, 4, 715, 8, 118, 14, 394, 20, 13, 119, 954, 189, 102, 5, 207, 110, 21, 14, 69, 188, 8, 30, 23, 7, 4, 249, 126, 93, 4, 114, 9, 5, 647, 4, 116, 9, 35, 4, 229, 9,

In [8]:
np.random.seed(42)
embed_size = 50
embed = np.random.rand(len(vocab), embed_size)
recurrent = np.eye(embed_size)
start = np.zeros(embed_size)
decoder = np.random.rand(embed_size, len(vocab))
one_hot = np.eye(len(vocab))

def predict(sent):
    layers = list()
    layer = {}
    layer['hidden'] = start
    layers.append(layer)
    loss = 0
    preds = list()

    for target_i in range(len(sent)):
        layer = {}
        layer['pred'] = softmax(layers[-1]['hidden'].dot(decoder))
        layer['hidden'] = layers[-1]['hidden'].dot(recurrent) + embed[sent[target_i]]
        layers.append(layer)
    return layers, loss

In [9]:
alpha = 0.001
for iter in range(1000):
    sent = tokens[iter % len(tokens)][1:]
    layers, loss = predict(sent)
    for layer_idx in reversed(range(len(layers))):
        layer = layers[layer_idx]
        target = sent[layer_idx - 1]
        if (layer_idx > 0):
            layer['output_delta'] = layer['pred'] - one_hot[target]
            new_hidden_delta = layer['output_delta'].dot(decoder.transpose())
            if (layer_idx == len(layers) - 1):
                layer['hidden_delta'] = new_hidden_delta
            else:
                layer['hidden_delta'] = new_hidden_delta + layers[layer_idx + 1]['hidden_delta'].dot(recurrent.transpose())
        else:
            layer['hidden_delta'] = layers[layer_idx+1]['hidden_delta'].dot(recurrent.transpose())

start -= layers[0]['hidden_delta'] * alpha / float(len(sent))
for layer_idx, layer in enumerate(layers[1:]):
    decoder -= np.dot(layers[layer_idx]['hidden'].reshape(-1, 1),layer['output_delta'].reshape(1, -1)) * alpha / float(len(sent))
    embed_idx = sent[layer_idx]
    embed[embed_idx] -= layers[layer_idx]['hidden_delta'] * alpha / float(len(sent))
    recurrent -= np.dot(layers[layer_idx]['hidden'].reshape(-1, 1),layer['hidden_delta'].reshape(1, -1)) * alpha / float(len(sent))

sent_index = 4
l, _ = predict(tokens[sent_index])
print(tokens[sent_index])
for i, each_layer in enumerate(l[1:-1]):
    input = tokens[sent_index][i]
    true = tokens[sent_index][i+1]
    pred = vocab[each_layer['pred'].argmax()]
    print("Prev Input:" + str(input) + (" " * (12 - len(str(input)))) + "True:" + str(true) + (" " * (15 - len(str(true)))) + "Pred:" + str(pred))

[1, 249, 7, 61, 113, 10, 10, 13, 14, 20, 56, 33, 18, 457, 88, 13, 45, 13, 70, 79, 49, 706, 919, 13, 16, 355, 340, 355, 96, 143, 4, 22, 32, 289, 7, 61, 369, 71, 5, 13, 16, 131, 249, 114, 249, 229, 249, 20, 13, 28, 126, 110, 13, 473, 8, 569, 61, 419, 56, 429, 6, 18, 35, 534, 95, 474, 570, 5, 25, 124, 138, 88, 12, 421, 52, 725, 61, 419, 11, 13, 15, 20, 11, 4, 2, 5, 296, 12, 5, 15, 421, 128, 74, 233, 334, 207, 126, 224, 12, 562, 298, 7, 5, 516, 988, 43, 8, 79, 120, 15, 595, 13, 784, 25, 18, 165, 170, 143, 19, 14, 5, 6, 226, 251, 7, 61, 113]
Prev Input:1           True:249            Pred:896
Prev Input:249         True:7              Pred:836
Prev Input:7           True:61             Pred:128
Prev Input:61          True:113            Pred:487
Prev Input:113         True:10             Pred:128
Prev Input:10          True:10             Pred:487
Prev Input:10          True:13             Pred:128
Prev Input:13          True:14             Pred:487
Prev Input:14          True:20           